# CS313 Nhóm 5 – E4: ablation DAMR (Bảng 3, 4), 3 dataset × 5 biến thể × 3 seed × 40 epoch

Biến thể (`scripts/run_e4.sh`): `wo_MSASB`, `wo_MSASB_G`, `wo_MSASB_L`, `fixed_tau_0.25`, `wo_temp_multihead`.
Bản `full` là E2 (seed 16, 17, 18). Cách hiểu MSASB(G)/(L) và τ của biến thể cuối: xem mục 12 `docs/code_vs_paper.md`.

**Trước khi chạy:**
1. *Settings → Accelerator*: **GPU T4 x1**. *Settings → Internet*: **On**.
2. *Add Input*: dataset chứa `psm/`, `msl/`, `smd/` (ví dụ `cs313-damr-data`).
3. Kiểm tra `BRANCH`, `DATASETS`, `VARIANTS` ở ô dưới.
4. Chạy bằng **Save Version → Save & Run All (Commit)**. Thời gian ước lượng: PSM, MSL mỗi cái ~1 giờ; SMD ~5 giờ.
   Nếu quá 12 giờ, chia làm 2 lần: lần 1 `DATASETS = ["PSM", "MSL"]`, lần 2 `["SMD"]`.

**Khi xong:** tab *Output* → tải `e4_outputs.zip`.


In [ ]:
REPO = "https://github.com/Waterbear566/cs313-damr-reproduce.git"
BRANCH = "main"
DATASETS = ["PSM", "MSL", "SMD"]
SEEDS = "16 17 18"           # seed chung của nhóm
VARIANTS = "wo_MSASB wo_MSASB_G wo_MSASB_L fixed_tau_0.25 wo_temp_multihead"
!rm -rf /kaggle/working/repo && git clone -q -b $BRANCH $REPO /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1

## 1. Môi trường

In [ ]:
# Kaggle đã có torch, numpy, pandas, scikit-learn. Chỉ cài phần còn thiếu, KHÔNG cài lại torch.
!pip install -q thop faiss-cpu lightning timm 2>&1 | tail -2
import platform, subprocess, torch, numpy, sklearn, pandas
print("Python   ", platform.python_version())
print("PyTorch  ", torch.__version__, "| CUDA", torch.version.cuda, "| cuDNN", torch.backends.cudnn.version())
print("numpy    ", numpy.__version__, "| pandas", pandas.__version__, "| sklearn", sklearn.__version__)
print("GPU      ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
assert torch.cuda.is_available(), "Chưa bật GPU: Settings → Accelerator → GPU T4"
!pip freeze > /kaggle/working/environment_kaggle.txt

## 2. Dữ liệu: tìm file trong /kaggle/input và nối vào `DAMR/data/`

In [ ]:
import glob, os
NEED = {"psm": ["train.csv", "test.csv", "test_label.csv"],
        "msl": ["MSL_train.npy", "MSL_test.npy", "MSL_test_label.npy"],
        "smd": ["SMD_train.npy", "SMD_test.npy", "SMD_test_label.npy"]}
for ds, files in NEED.items():
    if ds.upper() not in DATASETS:
        continue
    cands = [d for d in {os.path.dirname(p) for p in glob.glob(f"/kaggle/input/**/{files[0]}", recursive=True)}
             if all(os.path.exists(os.path.join(d, f)) for f in files)]
    if ds == "psm":   # tên train.csv rất phổ biến: ưu tiên thư mục có chữ psm
        cands = sorted(cands, key=lambda d: "psm" not in d.lower())
    assert cands, f"{ds.upper()}: không thấy trong /kaggle/input, kiểm tra Add Input"
    os.makedirs(f"DAMR/data/{ds}", exist_ok=True)
    for f in files:
        dst = f"DAMR/data/{ds}/{f}"
        if not os.path.exists(dst):
            os.symlink(os.path.join(cands[0], f), dst)
    print(f"{ds.upper()}: dùng {cands[0]}")
DS_ARGS = " ".join(DATASETS)
!python scripts/check_data.py --root DAMR/data --datasets $DS_ARGS

## 3. Chạy E4
Mỗi dataset chạy mọi biến thể × 3 seed bằng `scripts/run_e4.sh`. Log ở `logs/E4_<dataset>_<biến thể>_seed<k>.log`.


In [ ]:
import time
os.environ["SEEDS"] = SEEDS
os.environ["VARIANTS"] = VARIANTS
timing = {}
for ds in DATASETS:
    t0 = time.time()
    !bash scripts/run_e4.sh {ds} 0
    timing[ds] = (time.time() - t0) / 60
    print(f"===== {ds} xong sau {timing[ds]:.1f} phút =====")
print(timing)

## 4. Tổng hợp: mean ± std, `best_test` (như code gốc) và `last` (epoch cuối)

In [ ]:
!python scripts/summarize.py results/E4_DAMR_ablation.csv --seeds 16 17 18 --out results/E4_summary.csv


## 5. Đóng gói để tải về (tab Output → `e2_outputs.zip`)

In [ ]:
import shutil, zipfile
shutil.copy("/kaggle/working/environment_kaggle.txt", "results/")
with zipfile.ZipFile("/kaggle/working/e4_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for top in ["results", "logs", "checkpoints/E4"]:
        for root, _, files in os.walk(top):
            for f in files:
                z.write(os.path.join(root, f))   # đường dẫn tương đối gốc repo, dấu /
print(f"e4_outputs.zip: {os.path.getsize('/kaggle/working/e4_outputs.zip') / 1e6:.1f} MB")
!du -sh checkpoints/E4/*